# Health Risk Classification (Diabetes)
Name: Ali Siddiqui  
Roll No: 24f-AI-095  
Section: B  
Lab 7 - Open Ended Lab

In [ ]:
# Health Risk Classification | Ali Siddiqui | 24f-AI-095 | Section B
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings("ignore")

**Dataset:** Pima Indians Diabetes Database, 768 patients, 8 features (Pregnancies, Glucose, BloodPressure, SkinThickness, Insulin, BMI, DiabetesPedigreeFunction, Age). The target `Outcome` is 1 if the patient is diabetic and 0 if not.

In [ ]:
df = pd.read_csv("diabetes.csv")
print(df.shape)
print(df["Outcome"].value_counts())
df.describe()

In [ ]:
# checking which features are related to outcome
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(), annot=True, cmap="coolwarm")
plt.show()

**Preprocessing:** split into train/test, replace the impossible zeros with the median, then scale the data (needed for SVM).

In [ ]:
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# 80/20 split, stratify so both have same ratio of diabetic patients
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
# 0 can't be real for these columns so treating them as missing
# and filling with the median of the train data
cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

for c in cols:
    X_train[c] = X_train[c].replace(0, np.nan)
    X_test[c] = X_test[c].replace(0, np.nan)
    med = X_train[c].median()
    X_train[c] = X_train[c].fillna(med)
    X_test[c] = X_test[c].fillna(med)

print(X_train.isnull().sum().sum(), "missing values left")

In [ ]:
# scaling for svm (tree doesn't need it so it will use the normal data)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
# first try both models with default settings
svm = SVC()
svm.fit(X_train_s, y_train)

tree = DecisionTreeClassifier(random_state=42)
tree.fit(X_train, y_train)

print("SVM  train:", round(svm.score(X_train_s, y_train), 3), "test:", round(svm.score(X_test_s, y_test), 3))
print("Tree train:", round(tree.score(X_train, y_train), 3), "test:", round(tree.score(X_test, y_test), 3))

The tree gets 100% on train but much lower on test, so it is overfitting. Trying different settings with cross validation to fix this.

In [ ]:
# trying different C values for svm
for C in [0.1, 1, 10, 100]:
    score = cross_val_score(SVC(C=C), X_train_s, y_train, cv=5).mean()
    print("SVM C =", C, "cv accuracy =", round(score, 3))

In [ ]:
# trying different max_depth values for the tree
for d in [2, 3, 4, 5, 6, 8, 10]:
    score = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=42), X_train, y_train, cv=5).mean()
    print("Tree max_depth =", d, "cv accuracy =", round(score, 3))

In [ ]:
# final models using best values from above
svm2 = SVC(C=1, probability=True, random_state=42)
svm2.fit(X_train_s, y_train)

tree2 = DecisionTreeClassifier(max_depth=4, random_state=42)
tree2.fit(X_train, y_train)

svm_pred = svm2.predict(X_test_s)
tree_pred = tree2.predict(X_test)

print("SVM test accuracy :", round(accuracy_score(y_test, svm_pred), 3))
print("Tree test accuracy:", round(accuracy_score(y_test, tree_pred), 3))
print()
print("SVM")
print(classification_report(y_test, svm_pred))
print("Decision Tree")
print(classification_report(y_test, tree_pred))

In [ ]:
# confusion matrix of both models
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.heatmap(confusion_matrix(y_test, svm_pred), annot=True, fmt="d", cmap="Blues")
plt.title("SVM")

plt.subplot(1, 2, 2)
sns.heatmap(confusion_matrix(y_test, tree_pred), annot=True, fmt="d", cmap="Blues")
plt.title("Decision Tree")

plt.show()

In [ ]:
# overfitting check for the tree, train vs test accuracy for different depths
train_acc = []
test_acc = []
depths = range(1, 16)

for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=42)
    t.fit(X_train, y_train)
    train_acc.append(t.score(X_train, y_train))
    test_acc.append(t.score(X_test, y_test))

plt.plot(depths, train_acc, label="train")
plt.plot(depths, test_acc, label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.legend()
plt.show()

In [ ]:
# insulin seemed to be the most important feature for the tree so checking it
imp = pd.Series(tree2.feature_importances_, index=X.columns)
print(imp.sort_values(ascending=False).round(3))

# most common insulin value for each class
print()
print(df[df["Outcome"] == 0]["Insulin"].value_counts().head(1))
print(df[df["Outcome"] == 1]["Insulin"].value_counts().head(1))

In [ ]:
# testing again without insulin to see how much it matters
Xtr = X_train.drop(columns=["Insulin"])
Xte = X_test.drop(columns=["Insulin"])

sc = StandardScaler()
svm3 = SVC(C=1).fit(sc.fit_transform(Xtr), y_train)
tree3 = DecisionTreeClassifier(max_depth=4, random_state=42).fit(Xtr, y_train)

print("SVM without insulin :", round(svm3.score(sc.transform(Xte), y_test), 3))
print("Tree without insulin:", round(tree3.score(Xte, y_test), 3))

In [ ]:
# trying a new patient (made up values)
patient = pd.DataFrame([[2, 150, 80, 30, 160, 34.0, 0.6, 45]], columns=X.columns)

p1 = svm2.predict_proba(scaler.transform(patient))[0][1]
p2 = tree2.predict_proba(patient)[0][1]
print("SVM risk:", round(p1 * 100, 1), "%")
print("Tree risk:", round(p2 * 100, 1), "%")

**Conclusion**

- Both models gave about the same test accuracy (83.8%). SVM was a bit better at catching diabetic patients (recall 0.78 vs 0.74 for the tree).
- The tree with no limit got 100% on train but only about 81% on test, so it was overfitting. Setting max_depth to 4 (best cv accuracy) fixed most of it.
- SVM worked best with C = 1, bigger C values got worse in cross validation.
- Insulin is the most important feature by far, but in this csv the missing insulin values were filled with a different number for each class (102.5 and 169.5), so it basically leaks the answer. Without insulin accuracy drops to about 75% (SVM) and 78% (tree), so the real accuracy is probably lower than what we got.